# Automated Hyperparameter Optimization and Model Selection for NLP Pipelines

*NLPAICS 2026 Summer School - Ernesto Luis Estevanell*

Run one text-classification problem through three levels of automation: a hand-built scikit-learn model, a scikit-learn `Pipeline`, and AutoGOAL searches over full pipelines.

Part 4 adds a custom MiniLM embedding algorithm. As you run, track three things: what data each method is allowed to see, what search space we gave it, and whether the extra automation helped within the budget.


## Getting set up

One cell to confirm the kernel is the right one and AutoGOAL is importable. If it fails, the message tells you how to fix it, almost always *Kernel > Change Kernel > "NLPAICS 07"*.

In [ ]:
# --- Sanity check: right kernel, AutoGOAL present -----------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
# Run fully offline: the model + data are pre-cached by setup.sh, so we never let
# HuggingFace phone home mid-session (a algorithming call that stalls on conference wifi).
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
# macOS only: let AutoGOAL's worker processes fork safely (no-op on Linux/vast.ai)
os.environ.setdefault("OBJC_DISABLE_INITIALIZE_FORK_SAFETY", "YES")

if sys.version_info[:2] not in [(3, 9), (3, 10), (3, 11), (3, 12)]:
    raise SystemExit(f"Kernel is Python {sys.version.split()[0]}, AutoGOAL needs 3.9 to 3.12. "
                     "Fix: Kernel > Change Kernel > 'NLPAICS 07'.")
try:
    import autogoal
    from autogoal_contrib import find_classes
except ModuleNotFoundError as e:
    raise SystemExit(f"AutoGOAL not in this kernel (missing '{e.name}'). "
                     "Pick the 'NLPAICS 07' kernel and make sure ./setup.sh 07 finished.")

# AutoGOAL runs each candidate pipeline in a forked worker process; this line just
# makes that safe and identical on Linux/vast.ai and macOS (you can ignore it).
import multiprocessing as mp
try:
    mp.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass

algos = find_classes()
print("Python", sys.version.split()[0])
print(f"AutoGOAL ready -- {len(algos)} algorithms available.")

## The data: 20 Newsgroups

Everything in this notebook uses one four-topic slice of 20 Newsgroups: `rec.autos`, `sci.med`, `sci.space`, and `talk.politics.misc`.

We strip headers, footers, and quotes so the model has to learn from message text rather than metadata shortcuts.


In [ ]:
# --- Load the four-topic slice (cached by setup.sh) ---------------------------
from collections import Counter
from sklearn.datasets import fetch_20newsgroups

CATEGORIES = ['rec.autos', 'sci.med', 'sci.space', 'talk.politics.misc']
strip = ('headers', 'footers', 'quotes')   # drop metadata giveaways; learn the language
train = fetch_20newsgroups(subset='train', categories=CATEGORIES, remove=strip, random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES, remove=strip, random_state=0)

print(f"train: {len(train.data)} docs    test: {len(test.data)} docs    classes: {len(train.target_names)}")
print("classes:", train.target_names)

# one real example, so we know what a "document" looks like
ex = next(i for i, d in enumerate(train.data) if len(d.strip()) > 60)
print("\n--- an example document (first 300 chars) ---\n")
print(train.data[ex][:300].strip(), "...")
print("\nlabel:", train.target_names[train.target[ex]])

In [ ]:
# --- A quick look: how big, and how balanced? --------------------------------
import numpy as np
import matplotlib.pyplot as plt

names = train.target_names
tr = Counter(train.target); te = Counter(test.target)
print("class balance (train):", {names[i]: tr[i] for i in range(len(names))})

# how long are the posts? (words, after stripping metadata)
lengths = np.array([len(d.split()) for d in train.data])
print(f"document length (words): median {int(np.median(lengths))}, "
      f"mean {lengths.mean():.0f}, max {lengths.max()}  "
      f"({(lengths == 0).sum()} posts went empty after stripping)")

x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, [tr[i] for i in range(len(names))], w, label=f'train (n={len(train.data)})')
ax.bar(x + w/2, [te[i] for i in range(len(names))], w, label=f'test (n={len(test.data)})')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('documents'); ax.set_title('20 Newsgroups (4 topics): class balance')
ax.legend(); ax.grid(axis='y', alpha=0.3); fig.tight_layout(); plt.show()

### How we'll use the data

Use the training set for fitting and search decisions. Use the test set only for final checks.

Search methods create their validation signal from the training data: cross-validation for scikit-learn, an internal holdout for AutoGOAL.


## Part 1: Tuning a classifier with scikit-learn

A text classifier has two pieces: a **vectorizer** that turns text into numbers, and a **classifier** that learns from those numbers.

We first wire them by hand, then improve the representation with a few common NLP choices.

*(The systematic way to sweep those settings, grid and random search, lives in an [appendix](#appendix) at the end, so the session can spend its time on the AutoML payoff.)*


In [ ]:
# --- 1.1  The manual baseline: pick a vectorizer and a classifier by hand ------
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

vec = CountVectorizer()
X_train = vec.fit_transform(train.data)        # fit the vocabulary on train only
X_test  = vec.transform(test.data)

clf = LinearSVC(max_iter=5000, random_state=0)
clf.fit(X_train, train.target)
BASELINE_ACC = accuracy_score(test.target, clf.predict(X_test))

print(f"vocabulary learned : {len(vec.vocabulary_):,} words")
print(f"training matrix    : {X_train.shape[0]} docs x {X_train.shape[1]:,} features")
print(f"baseline (CountVectorizer -> LinearSVC), test accuracy = {BASELINE_ACC:.1%}")

That score is our starting point: one vectorizer, one classifier, every choice made by hand. Holding those choices by hand is exactly the manual work AutoGOAL automates later.

The first lever is the representation. NLP gives us standard pieces for it (tokenizing, dropping stopwords, stemming), and it pays to see them as discrete, swappable components, because in Part 3 AutoGOAL treats them as exactly that.

In [ ]:
# --- 1.2  The representation steps, as components AutoGOAL will later compose ----
# Tokenize, drop stopwords, stem: NLTK exposes each as an explicit step. AutoGOAL's
# registry wraps versions of exactly these, and the search decides which to use.
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

doc     = next(d for d in train.data if 60 < len(d.strip()) < 200)   # a short, real post
stop_en = set(stopwords.words('english'))
stem    = PorterStemmer()
content = [t for t in word_tokenize(doc) if t.isalpha() and t.lower() not in stop_en]

print("RAW          :", repr(doc.strip()[:100]))
print("no stopwords :", content[:12])
print("stemmed      :", [stem.stem(t) for t in content][:12])

scikit-learn bakes some of these into its vectorizers; NLTK keeps them as separate, swappable steps. Either way they are representation choices, and in Part 3 AutoGOAL searches over algorithms built from components like these.

In [ ]:
# --- 1.3  A better representation, still chosen by hand ------------------------
# TF-IDF weighting, bigrams, and stopword removal: three manual representation choices.
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
X_train = vec.fit_transform(train.data)
X_test  = vec.transform(test.data)

clf = LinearSVC(C=1, max_iter=5000, random_state=0)
clf.fit(X_train, train.target)
HAND_TUNED_ACC = accuracy_score(test.target, clf.predict(X_test))

print(f"hand-tuned (TF-IDF + bigrams + stopwords -> LinearSVC), test accuracy = {HAND_TUNED_ACC:.1%}")
print(f"gain over baseline = {HAND_TUNED_ACC - BASELINE_ACC:+.1%}")

Compare the printed scores. The model changed only because we changed the representation choices.

The next question is how to search those choices systematically instead of choosing them by hand.


## Part 2: Packaging the model as a `Pipeline`

So far we have carried the vectorizer and classifier as two separate objects. scikit-learn's `Pipeline` packages them into one estimator, with a single `.fit()` and `.predict()` over the whole chain. That matters here for one reason: a pipeline (vectorizer then classifier) is the unit AutoGOAL searches over in Part 3.

In [ ]:
# --- 2.1  The hand-tuned model as one Pipeline -------------------------------
from sklearn.pipeline import Pipeline

best_hand_tuned = Pipeline([
    ('vec', TfidfVectorizer(ngram_range=(1, 2), stop_words='english')),
    ('clf', LinearSVC(C=1, max_iter=5000, random_state=0)),
]).fit(train.data, train.target)

HAND_TUNED_ACC = best_hand_tuned.score(test.data, test.target)
print("pipeline steps:", [name for name, _ in best_hand_tuned.steps])
print(f"hand-tuned Pipeline, test accuracy = {HAND_TUNED_ACC:.1%}   (identical to 1.3)")
print("Keep this score for comparison with AutoGOAL.")


Same score as above, now packaged as one object.

The `step__parameter` naming convention is what lets scikit-learn search over pipeline settings. The appendix shows grid and random search; next we let AutoGOAL search over the pipeline structure too.


## Part 3: Searching over whole pipelines with AutoGOAL

In Parts 1 and 2 we fixed the pipeline shape by hand and stored its score in `HAND_TUNED_ACC`. AutoGOAL removes that last manual step: you describe the problem and a budget, and it searches over the pipeline itself, the choice of steps, their order, and their settings.

Everything in AutoGOAL comes down to four things you control:

1. **The task**, written as input and output types (text in, class labels out).
2. **The registry**, the list of algorithms it may use. This list *is* your search space.
3. **The budget**, how long it may search and how long any single pipeline may run.
4. **The metric**, what "good" means (accuracy by default).

Hand it those four, and the search proposes pipelines, scores each on a held-out slice of the training data, and steers toward what works. The building blocks come from scikit-learn and NLTK, and a type system keeps them from being wired together nonsensically.

### What Part 3 covers

We will set each of those four controls in turn, then search:

- [What an AutoGOAL algorithm is](#p3-algorithms): the typed building blocks, what "input type -> output type" means.
- [The task in types](#p3-task): stating text-in, labels-out once.
- [The registry](#p3-registry): the menu of algorithms, and how the list becomes the search space.
- [The metric](#p3-metric): accuracy by default, or your own objective in one line.
- [The playground](#p3-playground): build a registry, set the budget, run the search. This is your exercise.

Two things wait for later. **Part 4** teaches AutoGOAL brand-new algorithms and types of your own. And an [optional "under the hood"](#p3-internals) section opens up the grammar and pipeline graph that make the search possible; skip it on a first pass.

<a id="p3-algorithms"></a>
### What an AutoGOAL algorithm is

The new idea here is not the algorithms themselves (you know vectorizers and classifiers); it is that each one carries a machine-readable type contract: the input it accepts and the output it produces. A TF-IDF vectorizer advertises `Sentence -> sparse matrix`; LinearSVC advertises `matrix + labels -> labels`.

AutoGOAL connects two algorithms only when those types line up, so any pipeline it builds is guaranteed to run. That single mechanism is what lets it assemble pipelines on its own. You will see the contract printed as `input_types() -> output_type()` throughout Part 3, and in Part 4 you will write one yourself.

<a id="p3-task"></a>
### The task, written in types

We can state the whole job in one line of types:

`(Seq[Sentence], Supervised[VectorCategorical]) -> VectorCategorical`

`Seq[Sentence]` is a list of texts. `VectorCategorical` is class labels. `Supervised[...]` marks the training labels as ground truth, so the search never tries to produce them mid-pipeline.

In [ ]:
# --- 3.1  The vocabulary (imported once) and our task, pinned down ---------------
from autogoal.kb import (
    Seq, Sentence, Word, Document, Text,     # text granularity: a Word is a Sentence is a Document is a Text
    Supervised, VectorCategorical,           # labels in (Supervised) | class labels out
)
# Our task, once and for all -- reused by every search below:
INPUT  = (Seq[Sentence], Supervised[VectorCategorical])
OUTPUT = VectorCategorical
print("TASK:", INPUT, "->", OUTPUT, "\n")

# The friendly names are aliases -- handy when you read AutoGOAL's output:
from autogoal.kb import MatrixContinuousSparse, MatrixContinuousDense
print("legend:  VectorCategorical      ==", repr(VectorCategorical))
print("         MatrixContinuousSparse ==", repr(MatrixContinuousSparse))
print("         MatrixContinuousDense  ==", repr(MatrixContinuousDense))

# A classifier literally advertises the Supervised label as its 2nd input:
from autogoal_sklearn import LinearSVC
print("\nLinearSVC wiring:", LinearSVC.input_types(), "->", LinearSVC.output_type())

<a id="p3-registry"></a>
### The registry: your search space

The algorithms you hand the search **are** the search space. A registry is just a Python list of algorithm classes. A short, curated list keeps the search focused; a broad list can waste budget on algorithms that make no sense for the task.

The next cell imports a menu of algorithms and defines `show_registry`, a quick validator you can call on any registry you build.

A valid registry needs at least one vectorizer and one classifier, so there is a path from text to a label.

In [ ]:
# --- 3.2  The algorithm menu: import what you need, see how each one wires up ----
# Every name below is importable, and the imported class IS the wrapped AutoGOAL algorithm.
# Pick from these to build a registry in the playground cell. (The optional "under the hood"
# section lists the full installed set; these are the ones that fit our text-classification task.)
from autogoal.kb import build_pipeline_graph
from autogoal_sklearn import (
    TfidfVectorizer, CountVectorizer, HashingVectorizer,          # text -> features
    LinearSVC, LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron, MultinomialNB, ComplementNB,
    BernoulliNB, NearestCentroid, KNeighborsClassifier,
    DecisionTreeClassifier, ExtraTreeClassifier,                  # features -> a label
    TruncatedSVD, TfidfTransformer,                               # features -> features
    CountVectorizerTokenizeStem,                                  # NLTK-backed vectorizer
)
from autogoal_nltk import (
    TreebankWordTokenizer, ToktokTokenizer,      # tokenizers
    PorterStemmer, SnowballStemmer,              # stemmers
    StopwordRemover, TextLowerer,                # cleanup
)

MENU = {
    "VECTORIZERS  (text -> features)": [TfidfVectorizer, CountVectorizer, HashingVectorizer],
    "CLASSIFIERS  (features -> a label)": [
        LinearSVC, LogisticRegression, SGDClassifier, RidgeClassifier,
        PassiveAggressiveClassifier, Perceptron, MultinomialNB, ComplementNB,
        BernoulliNB, NearestCentroid, KNeighborsClassifier,
        DecisionTreeClassifier, ExtraTreeClassifier],
    "FEATURE TRANSFORMERS  (features -> features, an optional middle step)": [
        TruncatedSVD, TfidfTransformer],
    "NLTK tokenize/stem  (use the vectorizer WITH a tokenizer and a stemmer)": [
        CountVectorizerTokenizeStem, TreebankWordTokenizer, ToktokTokenizer,
        PorterStemmer, SnowballStemmer, StopwordRemover, TextLowerer],
}
for role, algorithms in MENU.items():
    print(role)
    for c in algorithms:
        print(f"    {c.__name__:28s} {c.input_types()}  ->  {c.output_type()}")
    print()

def show_registry(reg):
    # Print the wiring of a registry and check it forms at least one valid pipeline.
    if not reg:
        print("(empty registry: add at least one vectorizer and one classifier)"); return
    for c in reg:
        print(f"  {c.__name__:28s} {c.input_types()}  ->  {c.output_type()}")
    try:
        g = build_pipeline_graph(input_types=INPUT, output_type=OUTPUT, registry=reg)
        print(f"\n  valid search space: {g.graph.number_of_nodes()} nodes, "
              f"{g.graph.number_of_edges()} edges. Ready to search.")
    except Exception as e:
        print(f"\n  not a valid registry yet ({type(e).__name__}). You need a path from text to a "
              "label: at least one vectorizer AND one classifier. (The NLTK tokenize/stem "
              "vectorizer also needs a tokenizer and a stemmer in the registry.)")

<a id="p3-metric"></a>
### The metric you optimize

By default AutoGOAL optimizes accuracy. If you cared about something else, such as macro-F1 on an imbalanced problem, you pass it as a one-line function:

```python
from sklearn.metrics import f1_score
AutoML(..., objectives=lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro"))
```

AutoGOAL maximizes the objective, so higher-is-better metrics drop straight in; a loss would need a minus sign. We keep accuracy for this session, to compare directly against the hand-tuned pipeline.

<a id="p3-playground"></a>
### Your playground: build a registry and search

The next cell comes pre-filled with a starter registry, so you can run it as-is for your first search, then edit `my_registry` and re-run as often as you like. The search dials (budget, population size, and so on) live in the **SEARCH SETTINGS** cell right below, so you change them in one place. AutoGOAL scores each candidate pipeline on a held-out slice of the training data, so the number you watch climb is a **validation** score, not the test score.

Unlike the appendix's grid search, each pipeline here is scored on one held-out split (`cross_validation_steps=1`) to stay within budget: faster, but noisier.

> **Predict.** Will AutoGOAL beat the hand-tuned pipeline, match it, or spend too much of the budget exploring weak pipelines?
>
> While it runs, pipelines stream past. Gray error lines are pipelines that failed or timed out; the search skips them.

### Search settings, in one place

These five dials drive **every** AutoGOAL search in this notebook: the playground below and `run_search` in Part 4. Change a value, re-run this cell, then run any search.

- **`SEARCH_TIMEOUT`**, total time the search may run (the hard stop).
- **`EVAL_TIMEOUT`**, time a single pipeline may run before it is abandoned.
- **`POP_SIZE`**, how many candidate pipelines are kept per generation.
- **`LEARNING_RATE`**, how fast the search shifts toward what scored well.
- **`SELECTION`**, the top fraction of each generation used as parents.

During the live session, run one search at a time. Each can use its full budget (five minutes here), so back-to-back searches add up fast.


In [ ]:
# ==== SEARCH SETTINGS  (change here, re-run, then run any search below) ============
from autogoal.utils import Min, Sec   # Min == 60, Sec == 1, so 5 * Min reads as 300 seconds

SEARCH_TIMEOUT = 5 * Min     # total wall-clock budget per search (the hard stop)
EVAL_TIMEOUT   = 30 * Sec    # abandon a single pipeline once it runs longer than this
POP_SIZE       = 10          # candidate pipelines kept per generation
LEARNING_RATE  = 0.05        # how fast the search shifts toward winners (PESearch learning_factor)
SELECTION      = 0.2         # fraction of the population kept as parents each generation

print(f"search budget {SEARCH_TIMEOUT}s  |  per-pipeline cap {EVAL_TIMEOUT}s  |  "
      f"pop {POP_SIZE}  |  learning_rate {LEARNING_RATE}  |  selection {SELECTION}")

In [ ]:
# ===========================================================================
#  YOUR PLAYGROUND: build a registry, then run. Re-run as often as you like.
#  (The search dials live in the SEARCH SETTINGS cell just above.)
# ===========================================================================
from autogoal.ml import AutoML
from autogoal.search import ConsoleLogger

y_train = [train.target_names[i] for i in train.target]   # AutoGOAL wants string labels
y_test  = [train.target_names[i] for i in test.target]

# THE REGISTRY  (this *is* the search space). Pick names from the menu above.
# Pre-filled with a solid starter set: run it as-is, then add or remove names and re-run.
my_registry = [
    TfidfVectorizer, CountVectorizer,
    LinearSVC, LogisticRegression, MultinomialNB,
]

# ---- validate the registry, then search (scored on AutoGOAL's own validation split) ----
show_registry(my_registry)
if my_registry:
    automl = AutoML(
        input=INPUT, output=OUTPUT, registry=my_registry,
        search_iterations=10_000,                 # let the clock be the stop
        search_timeout=SEARCH_TIMEOUT,            # total budget        (from SEARCH SETTINGS)
        evaluation_timeout=EVAL_TIMEOUT,          # cap any one pipeline (from SEARCH SETTINGS)
        pop_size=POP_SIZE,                        # candidates kept per generation
        learning_factor=LEARNING_RATE,            # how fast the search shifts toward winners
        selection=SELECTION,                      # fraction kept as parents each generation
        cross_validation_steps=1,
        errors="ignore", random_state=0,
    )
    automl.fit(train.data, y_train, logger=ConsoleLogger())
    print("\nbest validation score:", automl.best_scores_)
    print("best pipeline so far:\n", automl.best_pipelines_[0])
else:
    print("\n(add some algorithms to my_registry above, then run again)")

In [ ]:
# --- 3.3  When you're happy with a registry, open the test set ONCE --------------
import numpy as np
if 'automl' not in globals():
    print("Run the playground above first, with a non-empty registry.")
else:
    pred = list(automl.predict(test.data))
    acc = np.mean([p == t for p, t in zip(pred, y_test)])
    print(f"AutoGOAL best pipeline, TEST accuracy = {acc:.1%}")
    if 'HAND_TUNED_ACC' in globals():
        print(f"(hand-tuned comparison: {HAND_TUNED_ACC:.1%})")

`best_scores_` is validation performance, not test performance. Because the budget is time-based, different machines may evaluate different numbers of candidates.

Iterate using validation. Open the test-set cell only when you are happy with the search space. Try smaller and larger registries: which ingredients are actually useful?

<a id="p3-internals"></a>
### Under the hood (optional): the grammar, the graph, the full catalogue

For the curious, here is the machinery behind the search: the typed grammar that generates pipelines, the pipeline graph it explores, and the complete list of installed algorithms. Feel free to skip; nothing later depends on this.

In [ ]:
# --- 3.4  Define a space, read it, sample from it -----------------------------
from autogoal.grammar import (generate_cfg, Sampler, Union,
                              ContinuousValue, CategoricalValue, BooleanValue, DiscreteValue)
from autogoal.utils import nice_repr

@nice_repr
class SVM:
    def __init__(self, C: ContinuousValue(0.1, 10),
                 kernel: CategoricalValue("linear", "rbf"),
                 shrinking: BooleanValue()):
        self.C, self.kernel, self.shrinking = C, kernel, shrinking

@nice_repr
class DTree:
    def __init__(self, max_depth: DiscreteValue(1, 10)):
        self.max_depth = max_depth

grammar = generate_cfg(Union("Classifier", SVM, DTree))   # "a Classifier is an SVM OR a DTree"
print(grammar, "\n")

s = Sampler(random_state=2)
print("four random configurations:")
for _ in range(4):
    print("   ", grammar.sample(sampler=s))

### The space of pipelines is a graph

Given our input and output types and a registry of algorithms, AutoGOAL can lay out every type-compatible pipeline as a graph. Each algorithm is a node; an edge means two algorithms fit together.

This lets us inspect the search space before running any search.


In [ ]:
# --- 3.5  The space of pipelines AutoGOAL searches (under the hood) -------------
# Given our task (INPUT -> OUTPUT) and a registry, AutoGOAL lays out every pipeline that
# type-checks as a graph. The graph you see here IS the space the search explores.
from autogoal.grammar import Sampler
sample_registry = [TfidfVectorizer, CountVectorizer, LinearSVC, LogisticRegression, MultinomialNB]
space = build_pipeline_graph(input_types=INPUT, output_type=OUTPUT, registry=sample_registry)
print("algorithms in this space:", sorted(a.__name__ for a in space.nodes()))
print("graph:", space.graph.number_of_nodes(), "nodes,", space.graph.number_of_edges(), "edges\n")
print("three pipelines sampled from the space:")
samp = Sampler(random_state=0)
for _ in range(3):
    print("  ", space.sample(sampler=samp), "\n")

### The full algorithm catalogue

`find_classes()` returns every algorithm AutoGOAL discovered, wrapped from scikit-learn and NLTK. Here they are grouped by the type each one outputs, which is really "what job it does". The text-classification subset you saw in the menu (3.2) is drawn from this list.

In [ ]:
# --- 3.6  The full catalogue: everything AutoGOAL discovered, grouped by OUTPUT type ----
# find_classes() returns every algorithm AutoGOAL wrapped from scikit-learn and NLTK. Any of
# them can also be imported by name (as the menu in 3.2 did); the imported class IS the very
# same wrapped algorithm. Here we group the full set by the TYPE each one produces, which is
# really "what job it does".
from collections import defaultdict
from autogoal_contrib import find_classes
import autogoal_sklearn, autogoal_nltk

all_algorithms = find_classes()
print(f"{len(all_algorithms)} algorithms total   |   "
      f"sklearn: {len(find_classes(modules=[autogoal_sklearn]))}   "
      f"nltk: {len(find_classes(modules=[autogoal_nltk]))}\n")

ROLE = {   # a human label for each output type we care about for text classification
    "Tensor[1, Categorical, Dense]": "CLASSIFIERS          (-> a class label)",
    "Tensor[2, Continuous, Sparse]": "VECTORIZERS          (text -> sparse features)",
    "Tensor[2, Continuous, Dense]" : "MATRIX TRANSFORMERS  (reduce / scale features)",
    "Seq[Word]"                    : "TOKENIZERS  [NLTK]   (text -> words)",
    "Stem"                         : "STEMMERS    [NLTK]   (word -> stem)",
}
by_out = defaultdict(list)
for c in all_algorithms:
    by_out[str(c.output_type())].append(c.__name__)
for out, role in ROLE.items():
    print(role)
    print("    " + ", ".join(sorted(by_out[out])) + "\n")
print("(The rest -- taggers, chunkers, regressors, clustering -- won't help classify a document.)")

### A reusable search for Part 4

Part 4 runs one more search, so bundle the fixed parts into `run_search`: task signature, time budget, progress log, and validation setup.


In [ ]:
# --- 3.7  run_search: the fixed parts in one place -------------------------------
# Same five dials as the playground, read from the SEARCH SETTINGS cell by default.
# Override any of them for a single call, e.g. run_search(reg, X, y, eval_timeout=60 * Sec).
from autogoal.ml import AutoML
from autogoal.search import ConsoleLogger   # plain scrolling log -- robust on flaky wifi
                                            # (swap in RichLogger() for the live table)
def run_search(reg, X, y, *, timeout=None, eval_timeout=None, pop_size=None,
               learning_rate=None, selection=None, **overrides):
    am = AutoML(
        input=INPUT, output=OUTPUT, registry=reg,
        search_iterations=10_000,                              # the clock is the real stop
        search_timeout     = timeout       or SEARCH_TIMEOUT,  # total budget
        evaluation_timeout = eval_timeout  or EVAL_TIMEOUT,    # cap any single pipeline
        pop_size           = pop_size      or POP_SIZE,
        learning_factor    = learning_rate or LEARNING_RATE,
        selection          = selection     or SELECTION,
        cross_validation_steps=1, errors="ignore", random_state=0, **overrides,
    )
    am.fit(X, y, logger=ConsoleLogger())
    return am

In [ ]:
# --- CHECKPOINT 2, restart point for Part 4 (instant) ----------------------
# Jumping in here (or recovered from a restart)? This reloads everything Part 4 needs:
# INPUT/OUTPUT, the search settings, and run_search, so the rest runs standalone.
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1"); os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from sklearn.datasets import fetch_20newsgroups
from autogoal.ml import AutoML
from autogoal.search import ConsoleLogger
from autogoal.kb import Seq, Sentence, Supervised, VectorCategorical
from autogoal.utils import Min, Sec
from autogoal_contrib import find_classes

CATEGORIES = ['sci.med', 'sci.space', 'rec.autos', 'talk.politics.misc']
strip = ('headers', 'footers', 'quotes')
train = fetch_20newsgroups(subset='train', categories=CATEGORIES, remove=strip, random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES, remove=strip, random_state=0)
y_train = [train.target_names[i] for i in train.target]
y_test  = [train.target_names[i] for i in test.target]

INPUT  = (Seq[Sentence], Supervised[VectorCategorical])
OUTPUT = VectorCategorical

# Reuse the SEARCH SETTINGS from Part 3 if they exist; otherwise fall back to the same defaults.
try:
    SEARCH_TIMEOUT, EVAL_TIMEOUT, POP_SIZE, LEARNING_RATE, SELECTION
except NameError:
    SEARCH_TIMEOUT, EVAL_TIMEOUT, POP_SIZE, LEARNING_RATE, SELECTION = 5*Min, 30*Sec, 10, 0.05, 0.2

# (run_search re-defined here so Part 4 stands alone -- identical to 3.7)
def run_search(reg, X, y, *, timeout=None, eval_timeout=None, pop_size=None,
               learning_rate=None, selection=None, **overrides):
    am = AutoML(input=INPUT, output=OUTPUT, registry=reg, search_iterations=10_000,
                search_timeout=timeout or SEARCH_TIMEOUT, evaluation_timeout=eval_timeout or EVAL_TIMEOUT,
                pop_size=pop_size or POP_SIZE, learning_factor=learning_rate or LEARNING_RATE,
                selection=selection or SELECTION, cross_validation_steps=1, errors="ignore",
                random_state=0, **overrides)
    am.fit(X, y, logger=ConsoleLogger()); return am
print(f"Checkpoint 2 ready, {len(train.data)} train / {len(test.data)} test docs loaded.")

## Part 4: Teaching AutoGOAL something new

The registry is just a list of algorithm classes, so you can add your own.

An algorithm says what it takes in, what it gives back, and what settings it exposes to the search. We will write a tiny algorithm first, then wrap a pretrained MiniLM encoder as a real algorithm.


### A first, tiny algorithm

The smallest useful algorithm takes a sentence and returns a list of words. Subclass `AlgorithmBase`, type the `run` method, and annotate `__init__` settings you want AutoGOAL to search.


In [ ]:
# --- 4.1  The smallest possible custom algorithm: a typed transform ------------
from autogoal.kb import AlgorithmBase, Sentence, Seq, Word
from autogoal.grammar import BooleanValue, DiscreteValue
from autogoal.utils import nice_repr

@nice_repr
class LongWordTokenizer(AlgorithmBase):
    """Split a sentence into words, optionally lowercasing and dropping short tokens."""
    def __init__(self, min_length: DiscreteValue(0, 5), lower: BooleanValue()):
        self.min_length = min_length      # <- searchable: an integer in [0, 5]
        self.lower = lower                # <- searchable: True / False

    def run(self, input: Sentence) -> Seq[Word]:     # <- Sentence in, Seq[Word] out
        if self.lower:
            input = input.lower()
        return [w for w in input.split() if len(w) > self.min_length]

print("declared contract:", LongWordTokenizer.input_types(), "->", LongWordTokenizer.output_type())
print("example:", LongWordTokenizer(min_length=3, lower=True).run("The Apollo program reached the Moon"))

### Wrapping a transformer

MiniLM is an encoder: it turns text into dense vectors. That makes it a natural AutoGOAL algorithm, because its output can feed a classifier.

A generative LLM can also be wrapped as an algorithm, but you must decide its output type and how to parse it. The type contract is the same; the wrapping is more work.


In [ ]:
# --- 4.2  Wrap a transformer LM as an AutoGOAL algorithm ----------------------
import numpy as np
from autogoal.kb import AlgorithmBase, Seq, Sentence, MatrixContinuousDense
from autogoal.grammar import BooleanValue
from autogoal.utils import nice_repr

@nice_repr
class TransformerEmbedding(AlgorithmBase):
    """Pretrained MiniLM sentence embeddings as features: Seq[Sentence] -> (n × d) dense
    matrix (d = the model's embedding size: 384 for MiniLM, 768 for mpnet)."""
    _MODEL = None
    _NAME  = "all-MiniLM-L6-v2"
    _VEC   = {}   # cache ONE raw vector per unique document, so any subset is a cache hit

    def __init__(self, normalize: BooleanValue()):
        self.normalize = normalize

    @classmethod
    def _model(cls):
        if cls._MODEL is None:
            import torch
            from sentence_transformers import SentenceTransformer
            device = "cuda" if torch.cuda.is_available() else "cpu"   # use the vast.ai GPU if present
            cls._MODEL = SentenceTransformer(cls._NAME, device=device)
        return cls._MODEL

    def run(self, input: Seq[Sentence]) -> MatrixContinuousDense:
        # encode only documents we haven't seen; warming the full corpus once (in 4.2b) means
        # the forked search workers inherit a full cache and never re-load the model.
        missing = [d for d in input if d not in TransformerEmbedding._VEC]
        if missing:
            V = self._model().encode(missing, batch_size=64, show_progress_bar=False,
                                     convert_to_numpy=True).astype("float32")
            for d, v in zip(missing, V):
                TransformerEmbedding._VEC[d] = v
        raw = np.vstack([TransformerEmbedding._VEC[d] for d in input])
        if self.normalize:
            n = np.linalg.norm(raw, axis=1, keepdims=True); n[n == 0] = 1.0
            return raw / n
        return raw

print("contract:", TransformerEmbedding.input_types(), "->", TransformerEmbedding.output_type())

> **Predict the crossover.** As labelled data increases, does TF-IDF catch MiniLM early, late, or not at all?


In [ ]:
# --- 4.2b  Representation showdown, a learning curve (a *fair* comparison) --------
# Same classifier (LogisticRegression) on both representations, same held-out split,
# averaged over 3 random subsamples -- only the FEATURES differ, as we vary how much
# labelled data we have. (You can skip reading the val_acc helper; the payoff is the
# table + plot.)
import numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

y_all = [train.target_names[i] for i in train.target]
print(f"encoding {len(train.data)} docs with MiniLM on "
      f"{'cuda' if torch.cuda.is_available() else 'cpu'} (once)...", flush=True)
EMB_ALL = TransformerEmbedding(normalize=True).run(list(train.data))   # encode all docs once

def val_acc(n, seed):                      # <- plumbing; skip on first read
    idx = train_test_split(np.arange(len(train.data)), train_size=n,
                           random_state=seed, stratify=y_all)[0]
    y = np.array([y_all[i] for i in idx])
    tr, va = train_test_split(np.arange(n), test_size=0.25, random_state=seed, stratify=y)
    emb = LogisticRegression(max_iter=2000).fit(EMB_ALL[idx[tr]], y[tr]).score(EMB_ALL[idx[va]], y[va])
    docs = [train.data[i] for i in idx]
    tf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True)
    Xtr = tf.fit_transform([docs[j] for j in tr]); Xva = tf.transform([docs[j] for j in va])
    tfidf = LogisticRegression(max_iter=2000).fit(Xtr, y[tr]).score(Xva, y[va])
    return emb, tfidf

sizes = (200, 400, 800, 1500, 2200)
emb_mean, tfidf_mean = [], []
print(f"\n{'train docs':>10} | {'MiniLM-emb':>11} | {'TF-IDF':>9} | winner")
print("-" * 48)
for n in sizes:
    es, ts = zip(*[val_acc(n, s) for s in (0, 1, 2)])     # average over 3 subsamples
    e, t = np.mean(es), np.mean(ts)
    emb_mean.append(e); tfidf_mean.append(t)
    print(f"{n:>10} | {e:>10.1%} | {t:>8.1%} | {'MiniLM' if e > t else 'TF-IDF'}")

# the picture: where do the two lines cross?
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.plot(sizes, [v*100 for v in emb_mean],   "o-", label="MiniLM embeddings")
    plt.plot(sizes, [v*100 for v in tfidf_mean], "s-", label="TF-IDF")
    plt.xlabel("labelled training documents"); plt.ylabel("validation accuracy (%)")
    plt.title("Pretrained embeddings vs TF-IDF as data grows")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", type(e).__name__, e, ")")

Read the trend, not a single point. MiniLM starts with language knowledge from pretraining; TF-IDF starts from the labelled data in this notebook.

The useful question is not "which representation always wins?", but "which one helps in this data regime?"

To keep the comparison fair, the same classifier scores both representations on the same held-out split, averaged over repeated subsamples.


In [ ]:
# --- 4.3  Let AutoGOAL choose, with the transformer in the registry -----------
from autogoal_sklearn import (TfidfVectorizer, CountVectorizer,
                              LinearSVC, LogisticRegression, SGDClassifier)

# A low-data slice (800 docs) -- the regime where, above, the transformer was ahead.
N  = 800
Xs = list(train.data[:N])
ys = [train.target_names[i] for i in train.target][:N]

reg_llm = [TransformerEmbedding, TfidfVectorizer, CountVectorizer,
           LinearSVC, LogisticRegression, SGDClassifier]   # classic algorithms AND your algorithm
print("registry now includes your algorithm:",
      [getattr(c, "__name__", str(c)) for c in reg_llm])

# one line, budget + logging already baked in. Transformer pipelines are slower to evaluate,
# so give each one more headroom than the 30s default (the rest comes from SEARCH SETTINGS):
automl_llm = run_search(reg_llm, Xs, ys, eval_timeout=60 * Sec)
print("\nbest validation score:", automl_llm.best_scores_)
print("AutoGOAL picked:\n", automl_llm.best_pipelines_[0])

Check the printed pipeline above. Did AutoGOAL keep `TransformerEmbedding`, or did it prefer a classic text representation?

The exact choice can move between runs, so focus on whether the search can compare both families under the same typed-pipeline interface.


**Your turn, after class.** A few things worth trying on your own:

1. Swap the model: point `_NAME` at `"all-mpnet-base-v2"` (stronger, 768 dimensions) and re-run the curve. At home that is about a 420 MB download, so you would first unset the offline flags (`os.environ.pop("HF_HUB_OFFLINE", None)`) or pre-cache it in `setup.sh`.
2. Add a setting: give `TransformerEmbedding` a pooling choice (`CategoricalValue("mean", "max")`) and use it in `run`.
3. Write a little featurizer of your own, say counts of a few domain keywords, and add it to the registry.

### One more thing, if you are curious: your own type

The types are not fixed either. You can carve out a new one by subclassing an existing type and tightening what it matches. Here is an `Email` type that only accepts strings with an `@` in them; an algorithm that asked for `Seq[Email]` would then turn down anything else.


In [ ]:
# A custom semantic type, in five lines
from autogoal.kb import Text

class Email(Text):                       # Email ⊂ Text, automatically
    @classmethod
    def _match(cls, x):
        return super()._match(x) and "@" in x

print("isinstance('a@b.com', Email):", isinstance('a@b.com', Email))
print("isinstance('hello',   Email):", isinstance('hello', Email))
print("issubclass(Email, Text)     :", issubclass(Email, Text))

## Where we landed

Summary: we built a baseline, packaged it as a `Pipeline`, searched over full pipelines with AutoGOAL, and added a custom transformer algorithm.

The recurring pattern is the same: define the algorithms, budget, metric, and data boundary; let the search handle the tedious combinations.


<a id="appendix"></a>
## Appendix: Classical HPO with grid & random search

Earlier we improved the hand-built pipeline. This appendix shows the classical scikit-learn way to search a fixed pipeline's settings.

Run it after the notebook above (it reuses `train`/`test`), or read it on your own time.


In [ ]:
# --- A.1  Count the search space ----------------------------------------------
from math import prod
knobs = {
    'ngram_range':  [(1,1), (1,2), (1,3)],
    'min_df':       [1, 2, 5],
    'max_df':       [0.5, 0.75, 1.0],
    'stop_words':   [None, 'english'],
    'sublinear_tf': [False, True],
    'C':            [0.03, 0.1, 0.3, 1, 3, 10],
}
print("configurations in a full grid:", prod(len(v) for v in knobs.values()))
print("...and that is before we even consider a different classifier.")

The previous cell prints the size of the grid. Grid search tries combinations and keeps the best by cross-validation.

Before running, predict the direction and rough size of the change: small improvement, large improvement, or no improvement.


In [ ]:
# --- A.2  Grid search: try a representative slice, keep best by CV ------------
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([('vec', TfidfVectorizer()),
                 ('clf', LinearSVC(max_iter=5000, random_state=0))])

# The search space is a dict: 'stepname__hyperparameter' -> values to try (the same
# double-underscore naming from the Pipeline section). 'vec__' settings go to the
# vectorizer step, 'clf__' to the classifier -- this dict *is* the simplest way to say it.
grid = {'vec__ngram_range':  [(1,1), (1,2)],
        'vec__min_df':       [1, 2, 5],
        'vec__stop_words':   [None, 'english'],
        'vec__sublinear_tf': [False, True],
        'clf__C':            [0.1, 0.3, 1, 3, 10]}

gs = GridSearchCV(pipe, grid, cv=5, n_jobs=-1).fit(train.data, train.target)
print(f"grid: {len(gs.cv_results_['params'])} configs × 5-fold CV")
print(f"best cross-val accuracy = {gs.best_score_:.1%}")
print(f"TEST accuracy           = {gs.score(test.data, test.target):.1%}")
print("winning knobs:", gs.best_params_)

Compare the printed test score with the baseline output. Cross-validation can look slightly better than the final test check because it selected the best configuration from many candidates.

Grids grow quickly. Random search samples the space instead of enumerating it.


In [ ]:
# --- A.3  Random search: sample the space instead of enumerating it -----------
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# Same step__param dict, with one upgrade: a continuous knob can be a *distribution*
# to sample from, not just a fixed list. (We reuse `pipe` from A.2.)
space = {'vec__ngram_range':  [(1,1), (1,2), (1,3)],
         'vec__min_df':       [1, 2, 3, 5],
         'vec__max_df':       [0.5, 0.75, 1.0],
         'vec__stop_words':   [None, 'english'],
         'vec__sublinear_tf': [False, True],
         'clf__C':            loguniform(1e-2, 1e2)}      # a distribution, not a list

rs = RandomizedSearchCV(pipe, space, n_iter=40, cv=5, random_state=0, n_jobs=-1)
rs.fit(train.data, train.target)
print(f"random: 40 sampled configs × 5-fold CV")
print(f"best cross-val accuracy = {rs.best_score_:.1%}")
print(f"TEST accuracy           = {rs.score(test.data, test.target):.1%}")
print("winning knobs:", {k: (round(v,3) if isinstance(v,float) else v) for k,v in rs.best_params_.items()})

Compare random search with grid search using the printed scores and runtimes.

The lesson is not the exact value; it is that random search can cover useful settings without enumerating the whole grid.

What it does *not* do is question the shape of the pipeline: it still assumes a vectorizer then a classifier, with the algorithms you picked. Letting the search choose the steps themselves is exactly the leap we took with AutoGOAL.


**Your turn.** Right now the classifier is fixed. Make the classifier part of the search as well. In the cell below, compare the linear SVM against logistic regression and multinomial naive Bayes. Which one comes out ahead here?


In [ ]:
# YOUR TURN 1: make the *classifier* part of the search.
# GridSearchCV can vary a whole pipeline step, not just its numbers: put a LIST of
# classifier instances under the key 'clf' and it will try each one and rank them.
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

pipe2 = Pipeline([('vec', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
                  ('clf', LinearSVC(max_iter=5000, random_state=0))])

grid2 = {
    # TODO: 'clf': [ ...three classifiers to compare... ]   # NB needs TF-IDF (we have it)
    # TODO: optionally one vectorizer knob, e.g. 'vec__sublinear_tf': [False, True]
}

# Uncomment once grid2 is filled in:
# gs2 = GridSearchCV(pipe2, grid2, cv=5, n_jobs=-1).fit(train.data, train.target)
# print(f"best CV {gs2.best_score_:.1%} with {type(gs2.best_params_['clf']).__name__}")
# Compare this score with the earlier grid/random-search outputs.

Give it a try first. When you're ready, run the next cell to see one answer, and the full ranking of the three classifiers.

In [ ]:
# Solution to Your turn 1
grid2 = {
    'clf': [LinearSVC(max_iter=5000, random_state=0), LogisticRegression(max_iter=2000), MultinomialNB()],
    'vec__sublinear_tf': [False, True],
}
gs2 = GridSearchCV(pipe2, grid2, cv=5, n_jobs=-1).fit(train.data, train.target)
print(f"best CV = {gs2.best_score_:.1%}  with  {type(gs2.best_params_['clf']).__name__}")
print("full ranking:")
import numpy as np
for acc, p in sorted(zip(gs2.cv_results_['mean_test_score'], gs2.cv_results_['params']), reverse=True):
    print(f"  {acc:.1%}  {type(p['clf']).__name__:20s} sublinear_tf={p['vec__sublinear_tf']}")